In [1]:
!nvidia-smi

Mon Nov 16 01:18:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install --progress-bar on transformers accelerate bitsandbytes scipy pandas==2.2.3 tqdm gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 34.4 MB/s  0:00:01


In [12]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Đường dẫn thư mục riêng
WORK = "/content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/nlp_macsql"

# Tự động tạo thư mục nếu chưa có
os.makedirs(WORK, exist_ok=True)
print("Thư mục làm việc đã được thiết lập tại:", WORK)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thư mục làm việc đã được thiết lập tại: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/nlp_macsql


In [5]:
import json, random
from collections import defaultdict, Counter

# Tự động tải file Spider về nếu chưa có
if not (os.path.exists("/content/spider/dev.json") or os.path.exists("/content/spider_data/dev.json")):
    !gdown "1TqleXec_OykOYFREKKtschzY29dUcVAQ" -O /content/spider.zip
    !unzip -q -o /content/spider.zip -d /content

ROOT = next(p for p in ["/content/spider", "/content/spider_data"] if os.path.exists(p + "/dev.json"))
DB_DIR = ROOT + "/database"
print("ROOT =", ROOT)

# Chọn 200 câu hỏi ngẫu nhiên (phân bổ đều theo các database)
SEED, N = 42, 200
dev = json.load(open(ROOT + "/dev.json"))
by_db = defaultdict(list)
for i, d in enumerate(dev): by_db[d["db_id"]].append(i)

random.seed(SEED)
chosen = []
for db, ix in sorted(by_db.items()):
    quota = max(1, round(N * len(ix) / len(dev)))
    chosen += random.sample(ix, min(quota, len(ix)))
chosen = sorted(set(chosen))
if len(chosen) > N: chosen = sorted(random.sample(chosen, N))
elif len(chosen) < N:
    rest = [i for i in range(len(dev)) if i not in set(chosen)]
    chosen = sorted(chosen + random.sample(rest, N - len(chosen)))

# Lưu lại danh sách 200 câu này để dùng cho các bước sau
subset = [{"id": i, "db_id": dev[i]["db_id"], "question": dev[i]["question"], "gold_sql": dev[i]["query"]} for i in chosen]
json.dump(subset, open(WORK + "/dev_subset.json", "w"), ensure_ascii=False, indent=1)

print(len(subset), "câu |", len({x['db_id'] for x in subset}), "DB")

Downloading...
From (original): https://drive.google.com/uc?id=1TqleXec_OykOYFREKKtschzY29dUcVAQ
From (redirected): https://drive.google.com/uc?id=1TqleXec_OykOYFREKKtschzY29dUcVAQ&confirm=t&uuid=1d9dd173-cd47-41d2-b523-3553fb3e3a90
To: /content/spider.zip
100% 99.7M/99.7M [00:00<00:00, 210MB/s]
ROOT = /content/spider
200 câu | 20 DB


In [6]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
CFG_PATH = WORK + "/precision.json"
tok = AutoTokenizer.from_pretrained(MODEL_ID)

def load_model(mode):
    if mode == "fp16": return AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="cuda")
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="cuda")

if os.path.exists(CFG_PATH):
    MODE = json.load(open(CFG_PATH))["mode"]; model = load_model(MODE)
else:
    try: MODE = "fp16"; model = load_model(MODE)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache(); MODE = "4bit"; model = load_model(MODE)
    json.dump({"mode": MODE, "model": MODEL_ID}, open(CFG_PATH, "w"))
model.eval()
print("Chế độ:", MODE, "| VRAM:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Chế độ: fp16 | VRAM: 6.17 GB


In [7]:
import sqlite3
def db_path(db_id): return os.path.abspath(f"{DB_DIR}/{db_id}/{db_id}.sqlite")

def get_schema(db_id):
    conn = sqlite3.connect(f"file:{db_path(db_id)}?mode=ro", uri=True)
    rows = conn.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' AND sql IS NOT NULL").fetchall()
    conn.close()
    return "\n\n".join(r[0].strip() for r in rows)

schema_map = {db: get_schema(db) for db in {x["db_id"] for x in subset}}
print("Đã lấy schema cho", len(schema_map), "database.")

Đã lấy schema cho 20 database.


In [8]:
import re, time
from collections import Counter

SYSTEM = ("You are an expert SQLite assistant. Given a database schema and a question, write exactly one valid SQLite query. Output only the query inside a ```sql code block.")
def build_prompt(question, ddl):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": f"### Database schema\n{ddl}\n\n### Question\n{question}"}]
def build_refine_prompt(question, ddl, old_sql, error):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": f"### Database schema\n{ddl}\n\n### Question\n{question}\n\n### Old SQL\n```sql\n{old_sql}\n```\n\n### SQLite error\n{error}\n\nFix the SQL. Output only the corrected query inside a ```sql code block."}]
@torch.no_grad()
def generate_sql(messages, max_new_tokens=256):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    reply = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"```(?:sql)?\s*(.*?)```", reply, re.DOTALL | re.IGNORECASE)
    sql = (m.group(1) if m else reply).strip()
    return sql.split(";")[0].strip() if ";" in sql else sql

def execute_sandbox(sql, db_id, timeout=30):
    conn = sqlite3.connect(f"file:{db_path(db_id)}?mode=ro", uri=True)
    conn.text_factory = lambda b: b.decode(errors="ignore")
    t0 = time.time()
    conn.set_progress_handler(lambda: 1 if time.time() - t0 > timeout else 0, 100000)
    try:
        rows = conn.execute(sql).fetchall()
        return {"ok": True, "rows": rows, "error": "", "error_type": None}
    except Exception as e:
        low = str(e).lower()
        et = "schema" if ("no such column" in low or "no such table" in low) else "syntax"
        return {"ok": False, "rows": [], "error": str(e), "error_type": et}
    finally: conn.close()

def _norm(rows): return [tuple(round(x, 6) if isinstance(x, float) else x for x in r) for r in rows]
def compare_results(pred, gold, has_order_by):
    p, g = _norm(pred), _norm(gold)
    return p == g if has_order_by else Counter(map(repr, p)) == Counter(map(repr, g))
def classify(res, correct):
    if correct: return "correct"
    if not res["ok"]: return res["error_type"]
    if len(res["rows"]) == 0: return "empty"
    return "silent"

def execution_accuracy(pred_sql, item):
    res = execute_sandbox(pred_sql, item["db_id"]); g = gold_cache[item["id"]]
    ok = res["ok"] and compare_results(res["rows"], g["rows"], "order by" in item["gold_sql"].lower())
    return bool(ok), res

gold_cache = {it["id"]: execute_sandbox(it["gold_sql"], it["db_id"]) for it in subset}
bad = [(i, r["error"]) for i, r in gold_cache.items() if not r["ok"]]
print("Gold lỗi:", len(bad))

Gold lỗi: 0


In [9]:
from tqdm.auto import tqdm
MAX_TRY = 3

def load_records(path): return [json.loads(l) for l in open(path, encoding="utf-8")] if os.path.exists(path) else []

def run_experiment(items, ckpt, refine_on_empty=False, max_try=MAX_TRY, init_sql=None):
    done = {r["id"] for r in load_records(ckpt)}
    with open(ckpt, "a", encoding="utf-8") as f:
        for it in tqdm(items):
            if it["id"] in done: continue
            t0 = time.time(); db = it["db_id"]; ddl = schema_map[db]
            sql = init_sql[it["id"]] if init_sql else generate_sql(build_prompt(it["question"], ddl))
            attempts = []
            for t in range(max_try + 1):
                ok, res = execution_accuracy(sql, it)
                attempts.append({"try": t, "sql": sql, "error": res["error"], "correct": ok, "label": classify(res, ok)})
                need = (not res["ok"]) or (refine_on_empty and len(res["rows"]) == 0)
                if (not need) or t == max_try: break
                err = res["error"] if not res["ok"] else "no data selected"
                sql = generate_sql(build_refine_prompt(it["question"], ddl, sql, err))
            rec = {**it, "attempts": attempts, "seconds": round(time.time() - t0, 2)}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
    return load_records(ckpt)
print("Sẵn sàng.")

Sẵn sàng.


In [13]:
DBG = WORK + "/checkpoint_debug.jsonl"
if os.path.exists(DBG): os.remove(DBG)
torch.cuda.reset_peak_memory_stats()
dbg = run_experiment(subset[:20], DBG)

sec = [r["seconds"] for r in dbg]
print("Giây/mẫu trung bình:", round(sum(sec)/len(sec), 1))
print("VRAM đỉnh:", round(torch.cuda.max_memory_allocated()/1e9, 2), "GB")
print("EX baseline (20 mẫu):", sum(r["attempts"][0]["correct"] for r in dbg), "/ 20")
print("EX sau sửa  (20 mẫu):", sum(r["attempts"][-1]["correct"] for r in dbg), "/ 20")

  0%|          | 0/20 [00:00<?, ?it/s]

Giây/mẫu trung bình: 3.2
VRAM đỉnh: 6.23 GB
EX baseline (20 mẫu): 16 / 20
EX sau sửa  (20 mẫu): 16 / 20


In [14]:
MAIN = WORK + "/checkpoint_main.jsonl"
recA = run_experiment(subset, MAIN, refine_on_empty=False)
print("Đã xong:", len(recA), "/", len(subset))

  0%|          | 0/200 [00:00<?, ?it/s]

Đã xong: 200 / 200


In [15]:
import pandas as pd, math
def ex_at(recs, k): return sum(r["attempts"][min(k, len(r["attempts"]) - 1)]["correct"] for r in recs)
def wilson(c, n, z=1.96):
    p = c / n; d = 1 + z*z/n
    ctr = (p + z*z/(2*n)) / d
    half = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return 100*(ctr-half), 100*(ctr+half)

rows = []; base = ex_at(recA, 0); n = len(recA)
for k in range(MAX_TRY + 1):
    c = ex_at(recA, k); lo, hi = wilson(c, n)
    rows.append({"maxTry": k, "Đúng": f"{c}/{n}", "EX (%)": round(100*c/n, 2), "CI95 dưới": round(lo, 1), "CI95 trên": round(hi, 1), "+EX so baseline": round(100*(c-base)/n, 2)})
tbl = pd.DataFrame(rows); print(tbl.to_string(index=False))
tbl.to_csv(WORK + "/bang_tong_hop.csv", index=False, encoding="utf-8-sig")

 maxTry    Đúng  EX (%)  CI95 dưới  CI95 trên  +EX so baseline
      0 127/200    63.5       56.6       69.9              0.0
      1 130/200    65.0       58.2       71.3              1.5
      2 130/200    65.0       58.2       71.3              1.5
      3 130/200    65.0       58.2       71.3              1.5


In [16]:
from scipy.stats import binomtest
b = sum((not r["attempts"][0]["correct"]) and r["attempts"][-1]["correct"] for r in recA)
c = sum(r["attempts"][0]["correct"] and (not r["attempts"][-1]["correct"]) for r in recA)
p = binomtest(min(b, c), b + c, 0.5).pvalue if (b + c) > 0 else 1.0
print(f"sai→đúng={b} | đúng→sai={c} | p={p:.4f}")

sai→đúng=3 | đúng→sai=0 | p=0.2500


In [18]:
from google.colab import files
for f in ["bang_tong_hop.csv", "dev_subset.json", "checkpoint_main.jsonl"]:
    if os.path.exists(f"{WORK}/{f}"): files.download(f"{WORK}/{f}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
from collections import Counter

# Tên các loại lỗi
names = {"correct": "Đúng", "schema": "1. Ảo giác tên cột/bảng", "syntax": "2. Lỗi cú pháp", "empty": "3. Kết quả rỗng", "silent": "4. Lỗi logic ngầm"}

# Đếm lỗi ở lượt đầu tiên (Baseline) và lượt cuối (Sau Refiner)
truoc = Counter(r["attempts"][0]["label"] for r in recA)
sau = Counter(r["attempts"][-1]["label"] for r in recA)

print("--- PHÂN BỐ LỖI TRƯỚC KHI REFINER (Baseline) ---")
for k in names:
    print(f"{names[k]}: {truoc.get(k,0)} câu ({truoc.get(k,0)/2:.1f}%)")

print("\n--- PHÂN BỐ LỖI SAU KHI REFINER (Pipeline) ---")
for k in names:
    print(f"{names[k]}: {sau.get(k,0)} câu ({sau.get(k,0)/2:.1f}%)")

--- PHÂN BỐ LỖI TRƯỚC KHI REFINER (Baseline) ---
Đúng: 127 câu (63.5%)
1. Ảo giác tên cột/bảng: 27 câu (13.5%)
2. Lỗi cú pháp: 0 câu (0.0%)
3. Kết quả rỗng: 5 câu (2.5%)
4. Lỗi logic ngầm: 41 câu (20.5%)

--- PHÂN BỐ LỖI SAU KHI REFINER (Pipeline) ---
Đúng: 130 câu (65.0%)
1. Ảo giác tên cột/bảng: 19 câu (9.5%)
2. Lỗi cú pháp: 1 câu (0.5%)
3. Kết quả rỗng: 5 câu (2.5%)
4. Lỗi logic ngầm: 45 câu (22.5%)
